In [1]:
from ome_zarr.image import NgffImage, NgffMultiscales
from ome_zarr.scene import NgffScene
from ome_zarr_models._v06.coordinate_transforms import CoordinateSystem, CoordinateSystemIdentifier, Axis, Translation
from bffile import BioFile
from bioio import BioImage
import os
import numpy as np
from dask.distributed import Client, get_client
import napari
import xmltodict

from multiview_stitcher import spatial_image_utils as si_utils
from multiview_stitcher import fusion
from skimage import io

In [4]:
root = r'C:\Users\johan\Desktop\local_data\BiaPoL\autoSCAPE\20260317\10x\J'
columns = os.listdir(root)

In [3]:
try:
  client = get_client()
except ValueError:
  client = Client()

print(client.dashboard_link)

http://127.0.0.1:8787/status


In [8]:
viewer = napari.Viewer()
viewer_projections = napari.Viewer()

In [4]:
def find_in_nested_dict(data, key, value):
    """
    Recursively search for objects containing a specific key-value pair.
    Returns a list of all matching objects.
    """
    results = []
    
    def search(obj):
        if isinstance(obj, dict):
            if obj.get(key) == value:
                results.append(obj)
            for v in obj.values():
                search(v)
        elif isinstance(obj, list):
            for item in obj:
                search(item)
    
    search(data)
    return results  

In [ ]:
write_jobs = []

for column in columns:
	images = []
	transforms = []
	for file in os.listdir(os.path.join(root, column)):
		if file.endswith(".ome.tif"):
			filename = os.path.join(root, column, file)
			outfile_oz = os.path.join(root, column, file.split(".")[0] + ".ome.zarr")
			bf = BioImage(filename)

			scale = {
				"z": bf.ome_metadata.images[0].pixels.physical_size_z *1e6,
				"y": bf.ome_metadata.images[0].pixels.physical_size_y *1e6,
				"x": bf.ome_metadata.images[0].pixels.physical_size_x *1e6
			}

			axes_units = {
				"z": "micrometer",
				"y": "micrometer",
				"x": "micrometer",
			}
			
			# parse tile metadata
			metadata_file = os.path.join(root, column, "Metadata", f"{file.split('.')[0]}.xlif")
			metadata_dict = xmltodict.parse(open(metadata_file).read())
			tile_scan_info = find_in_nested_dict(metadata_dict, '@Name', 'TileScanInfo')[0]["Tile"]
	
			# n_series= bf.series_count()
			n_series = len(bf.scenes)

			if n_series > 1:
				sims = []
				for idx in range(n_series):
					bf.set_scene(bf.scenes[idx])
					dask_image = bf.dask_data
					# dask_image = bf.to_dask(series=idx)
					ts_info = tile_scan_info[idx]

					t_z = float(ts_info["@PosZ"]) * 1e6
					t_y = float(ts_info["@PosY"]) * 1e6
					t_x = float(ts_info["@PosX"]) * 1e6

					sims.append(
						si_utils.get_sim_from_array(
							dask_image,
							dims=["t", "c", "z", "y", "x"],
							scale=scale,
							translation={"z": t_z, "y": t_y, "x": t_x},
							transform_key="stage_metadata",
							)
					)

				image = fusion.fuse(sims, transform_key="stage_metadata", fusion_func=fusion.max_fusion)
			else:
				# image = bf.to_dask(series=0)
				image = bf.dask_data
				t_z = float(tile_scan_info["@PosZ"]) * 1e6
				t_y = float(tile_scan_info["@PosY"]) * 1e6
				t_x = float(tile_scan_info["@PosX"]) * 1e6

			ngff_image = NgffImage(
				image.squeeze(),
				dims=["z", "y", "x"],
				scale=scale,
				axes_units=axes_units,
				name=file.split('.')[0]
			)
			ngff_multiscales = NgffMultiscales(
				image=ngff_image,
				scale_factors=[
				{"z": 2, "y": 2, "x": 2},
				{"z": 4, "y": 4, "x": 4},
				{"z": 8, "y": 8, "x": 8},
				{"z": 16, "y": 16, "x": 16},
				],
			)
			images.append(ngff_multiscales)

			transform = Translation(
				translation=(t_z, t_y, t_x),
				input=CoordinateSystemIdentifier(
					path=ngff_multiscales.name,
					name=ngff_multiscales.metadata.coordinateSystems[0].name
				),
				output=CoordinateSystemIdentifier(
					name="world"
				)
			)
			transforms.append(transform)
	
	cs_world = CoordinateSystem(
		name="world",
		axes=(
			Axis(name="z", type="space", unit="micrometer"),
			Axis(name="y", type="space", unit="micrometer"),
			Axis(name="x", type="space", unit="micrometer"),
		)
	)

	ngff_scene = NgffScene(
		images=images,
		transformations=transforms,
		coordinate_systems=(
			cs_world,
		)
	)

	delayed_objects = ngff_scene.to_ome_zarr(
		os.path.join(root, column, "scene.ome.zarr"), compute=False
	)

	write_jobs += [img_jobs[-1] for img_jobs in delayed_objects]

	print(f"Finished column {column}")

Finished column 10
Finished column 11
Finished column 12
Finished column 13
Finished column 14
Finished column 15
Finished column 16
Finished column 17
Finished column 18
Finished column 7
Finished column 8
Finished column 9
Finished column Metadata


In [ ]:
_ = client.compute(write_jobs)

[<Future: pending, key: apply-efde381b-c7bb-4ca5-a85c-1fb99ad6f0f3>,
 <Future: pending, key: apply-bfcd6e60-70fc-4177-b1f5-d28d80810fac>,
 <Future: pending, key: apply-ae8838a9-8051-4a60-afb9-ee7e403d755e>,
 <Future: pending, key: apply-dee9a73e-9a5d-4652-9a66-7d3e9636c9c5>,
 <Future: pending, key: apply-f70e28cd-f711-49ed-8586-4695911c10c6>,
 <Future: pending, key: apply-aaa584b6-c4d1-497e-b46b-b471e53fea17>,
 <Future: pending, key: apply-43c759e3-a3a0-4866-9946-a7fb753e6a3e>,
 <Future: pending, key: apply-fb8d355b-7f34-4b3b-a190-8c466998dece>,
 <Future: pending, key: apply-dbf0e8b5-c918-4100-94a0-0ddee84bfcd1>,
 <Future: pending, key: apply-018f4af1-cc4b-48ab-8c46-0b7335981004>,
 <Future: pending, key: apply-c173c483-9f5b-4438-b6a3-f26ba67683f2>,
 <Future: pending, key: apply-bf2ecc8f-279a-4301-a597-9b101a58beaa>,
 <Future: pending, key: apply-cca551f7-c1f9-4366-9433-211d5080cac2>,
 <Future: pending, key: apply-bff92f76-6b81-417d-9350-5dfbd5ccc61b>,
 <Future: pending, key: apply-c44f

## Show scenes



In [9]:
for column in columns:
	viewer.open(os.path.join(root, column, "scene.ome.zarr"), plugin="napari-ome-zarr")

Root group {'ome': {'scene': {'coordinateTransformations': [{'type': 'translation', 'input': {'name': 'physical', 'path': 'P1'}, 'output': {'name': 'world'}, 'translation': [1360.0117, 46188.4402, 48583.5365]}, {'type': 'translation', 'input': {'name': 'physical', 'path': 'P2'}, 'output': {'name': 'world'}, 'translation': [1360.0117, 46258.5748, 50074.0541]}, {'type': 'translation', 'input': {'name': 'physical', 'path': 'P3'}, 'output': {'name': 'world'}, 'translation': [1360.0117, 47813.0421, 48576.574499999995]}, {'type': 'translation', 'input': {'name': 'physical', 'path': 'P4'}, 'output': {'name': 'world'}, 'translation': [1360.0117, 47913.028900000005, 50099.4176]}], 'coordinateSystems': [{'name': 'world', 'axes': [{'name': 'z', 'type': 'space', 'unit': 'micrometer'}, {'name': 'y', 'type': 'space', 'unit': 'micrometer'}, {'name': 'x', 'type': 'space', 'unit': 'micrometer'}]}]}, 'version': '0.6'}}
Scene.matches {'scene': {'coordinateTransformations': [{'type': 'translation', 'input

ValueError: Reader 'napari-ome-zarr' was selected to open ['C:\\Users\\johan\\Desktop\\local_data\\BiaPoL\\autoSCAPE\\20260317\\10x\\J\\Metadata\\scene.ome.zarr'], but returned no data.

In [18]:
layer = viewer.layers['scene.ome [1]']
max_projection = layer.data[0].min(axis=0)
viewer.add_image(max_projection, scale = layer.affine.scale[1:], translate=layer.affine.translate[1:], name="max projection")

<Image layer 'max projection [1]' at 0x14656604890>

In [ ]:
for layer in viewer.layers:
	projection = layer.data[0].min(axis=0).rechunk(640, 640)

	# iterate over chunks and save as individual images
	for idx in np.ndindex(projection.numblocks):
		chunk = projection.blocks[idx]
		if not chunk.shape == (640, 640):
			continue

		client.submit(io.imsave, os.path.join(r"C:\Users\johan\Desktop\local_data\BiaPoL\autoSCAPE\20260317\projections", f"{layer.name}_chunk_{idx}.tif"), chunk)

In [46]:
viewer_projections.layers.clear()
viewer_projections.add_image(chunk, contrast_limits=(p5.compute(), p95.compute()))

<Image layer 'chunk' at 0x1d8aa80ea20>

In [ ]:
viewer.layers[0].data[0].min(axis=0).rechunk(640, 640)

(4, 5)

In [35]:
p5.compute()

np.float64(243.0)

In [36]:
p95.compute()

np.float64(981.0)